# ANÁLISIS Y ESTANDARIZACIÓN DE PROVEEDORES

In [1]:
import sys
import os
project_root = os.path.dirname(os.getcwd()) 
sys.path.insert(0, project_root)
from utils.imports import *

### VERSIONES DE LIBRERIAS

Python: 3.10.19
pandas: 2.3.3
numpy: 2.2.6
matplotlib: 3.10.0
seaborn: 0.13.2
Pandas: 2.3.3
IPython: 8.38.0
rapidfuzz versión: 3.14.5


## PRODUCTOS ODOO - INTERFUERZA

In [2]:
notebook_dir = Path.cwd()
raw_path = notebook_dir.parent / 'data' / 'raw'
processed_path = notebook_dir.parent / 'data' / 'processed'
df_odoo = pd.read_csv(raw_path / 'ProductosOdoo.csv')
df_odoo_copy = df_odoo.copy()
df_int = pd.read_excel(raw_path / 'ProductosInterfuerza.xlsx')
df_int_copy = df_int.copy()

display(Markdown("### Productos Odoo"))
print(f"Total productos: {len(df_odoo)}")
display(df_odoo.head(5))
display(Markdown("### Productos Interfuerza"))
print(f"Total productos: {len(df_int)}")
display(df_int.head(5))

### Productos Odoo

Total productos: 14215


,id,name,barcode,default_code,x_studio_many2many_field_37q_1irl4uc58,categ_id,seller_ids
0,__export__.product_template_31022_88373f19,ADORNO HALLOWEN,7453066213137,CF00614/autoriza noris,NaN,Cumpleaños / Artículos de Fiesta,NaN
1,__export__.product_template_219315_4256fd8c,ARCHIVADOR VERTICAL 3 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
2,__export__.product_template_219316_dfced89f,ARCHIVADOR VERTICAL 4 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN
3,__export__.product_template_76612_602c2239,ARTICULOS DE NAVIDAD,93,9,NaN,Navidad / Bolas,NaN
4,__export__.product_template_29991_44f2fe53,ARTICULOS FIESTA,17,1,NaN,Cumpleaños / Artículos de Fiesta,NaN


### Productos Interfuerza

Total productos: 73681


,Id,UPC Code,Ubicacion,Item Number,Tipo,Nombre,Proveedor Principal,Marca,Pais de Origen,Punto de ReOrden,...,Matriz Padre,Matriz Hijo,Arancel,Material,UoM InStock,UoM Compra,UoM Venta,Tags,Peso,Detalle
0,PRO101109,7512155423106,BODEGA PRINCIPAL,A404/6917,PRODUCTO,GORRA FBI 7512155423106,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,SOMBRERO SHERIFF
1,PRO104891,40568000067,NaN,405686/25.95OFERTA5.00,PRODUCTO,VESTIDO LARGO DAMA CHINO,CAPI WORLDWIDE SA,ZZZZZ,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VESTIDO LARGO DAMA CHINO
2,PRO107818,7465376497664,NaN,9599/HB2473-18,PRODUCTO,BIRRETE GRADUACION NEGRO-AZUL TELA 7465376497664,OMER SA,GENERICO,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,VIRRETE GRADUACION TELA
3,PRO107822,6930180607994,NaN,AFD-799,PRODUCTO,NUMEROS LED CHICO AFD-799,SOLARTE DE PANAMA SA,FIESTAS DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NUMEROS LED CHICO AFD-799
4,PRO107852,60000001438,BODEGA PRINCIPAL,601430,PRODUCTO,ARREGLO GRADUACION GLOBOS,CORPORACION DAISY SA,DAISY,NaN,0,...,NO,NO,NaN,NaN,NaN,NaN,NaN,NaN,0.0,ARREGLO GRADUACION GLOBOS


## ANÁLISIS DE PRODUCTOS SIN PROVEEDOR 

In [3]:
df_odoo_copy, df_int_copy, sellers_diagnostic, sellers_dict, df_int_seller_codes = prepare_data_and_diagnose(
    df_odoo_copy=df_odoo_copy,
    df_int_copy=df_int_copy,
    target_column='seller_ids',
    entity_name='proveedores',
    int_code_col='UPC Code',
    int_entity_col='Proveedor Principal'
)

### Normalización de códigos

#### Muestras de códigos normalizados

,original_odoo,normalizado_odoo,original_int,normalizado_int
0,7453066213137,7453066213137,7512155423106,7512155423106
1,NaN,None,40568000067,40568000067
2,NaN,None,7465376497664,7465376497664
3,93,93,6930180607994,6930180607994
4,17,17,60000001438,60000001438


### Diagnóstico inicial

#### Diagnóstico

Total de productos: 14215
Con proveedores: 8664 (60.95%)
Sin proveedores: 5551 (39.05%)


#### Estadísticas

Total de proveedores con valores únicos: 222
proveedores más frecuente: PROV-ARGELIA INTERNACIONAL S A (851 productos)
proveedores menos frecuente: PATPHA SA (1 productos)
proveedores con un solo producto: 47 (21.17% del total)


##### Creación de diccionario código-proveedores

Mapa creado con 62396 códigos de barras únicos


### Identificación de códigos con múltiples proveedores

In [4]:
multiple_sellers, df_multiple_sellers = analyze_multiple_entities(
    df=df_int_seller_codes,
    code_col='barcode_norm',
    entity_col='Proveedor Principal',
    entity_name='proveedores',
    output_path=raw_path / 'MultiplesProveedores.csv'
)

#### Análisis de múltiples proveedores por código de barras


 ##### Estadísticas generales

Total registros en Interfuerza: 62,512
Total códigos de barras únicos: 62,396
Códigos con múltiples proveedores: 50
Porcentaje del total: 0.08%



 ##### Distribución de multiplicidad

Códigos con 2 proveedores diferentes: 50
Códigos con 3 proveedores diferentes: 0
Códigos con 4+ proveedores diferentes: 0


#### Muestra de códigos con múltiples proveedores

,barcode_norm,Proveedor Principal
0,10100001258,"[DISTRIBUIDORA OFFICE TOTAL, S.A, CENTRO AMERI..."
1,104142,"[INVERSIONES OJENISA SA, CORPORACION DAISY SA]"
2,10514000083,"[SUPRICOM SA, CORPORACION DAISY SA]"
3,105940,"[DESTREZA PANAMA SA, CORPORACION DAISY SA]"
4,105942,"[CORPORACION DAISY SA, MEGA ENSAMBLES INT S.A.]"
5,105944,"[DESTREZA PANAMA SA, LA VICTORIA, S.A. o ALMAC..."
6,105945,"[DESTREZA PANAMA SA, CORPORACION DAISY SA]"
7,105949,"[LA VICTORIA, S.A. o ALMACEN SUN WAH, CORPORAC..."
8,106019,"[DESTREZA PANAMA SA, CORPORACION DAISY SA]"
9,13578105725,"[CARMEL FARMACEUTICA, ABERNATHY S A]"


Archivo 'Proveedores.csv' generado


### Asignación de proveedores a productos sin marca

In [5]:
df_odoo_sellers_import, df_sellers_pending = assign_entities_to_products(
    df_products=df_odoo_copy,
    missing_mask=sellers_diagnostic,
    codes_to_exclude=multiple_sellers,  
    entity_dict=sellers_dict,
    entity_name='proveedor',
    id_col='id',
    target_col='seller_ids',
    output_path=processed_path / 'ProveedoresParaActualizar.csv'
)

#### Asignación de productos sin proveedor

Productos sin proveedor (excluyendo conflictivos): 5536
✅ Resueltos automáticamente: 5216 (94.2%)
❌ Pendientes (sin coincidencia): 320 (5.8%)




#### MUESTRA DE PRODUCTOS SIN PROVEEDOR PENDIENTES

,id,name,barcode,default_code,x_studio_many2many_field_37q_1irl4uc58,categ_id,seller_ids,barcode_norm,entidad_asignada
1,__export__.product_template_219315_4256fd8c,ARCHIVADOR VERTICAL 3 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN,None,NaN
2,__export__.product_template_219316_dfced89f,ARCHIVADOR VERTICAL 4 GAVETAS Y CERRADURA,NaN,NaN,NaN,All,NaN,None,NaN
7,__export__.product_template_219323_100f066b,BATERIA PARA MAQUINA SOPLADORA DE 20 V,NaN,NaN,NaN,All,NaN,None,NaN
9,__export__.product_template_219266_6f6f9bfc,CADENA STRASS SS6 PLATEADO X YDA,109225,CL:SLV-6-SS6,NaN,Sedería / Cintas,NaN,109225,NaN
11,__export__.product_template_219319_c420f23e,CARGADOR PARA MAQUINA SOPLADORA DE 20 V,NaN,NaN,NaN,All,NaN,None,NaN
15,__export__.product_template_219317_9a13e3bb,DESMALEZADORA MULTI-HERRAMIENTA PARA PODAR,NaN,NaN,NaN,All,NaN,None,NaN
16,__export__.product_template_219076_4e78e199,ESCRITORIO PARA DOCENTE,NaN,NaN,NaN,All,NaN,None,NaN
22,__export__.product_template_219074_89b85cc6,SILLA ESCOLAR BRAZO DERECHO,NaN,NaN,NaN,All,NaN,None,NaN
23,__export__.product_template_219075_2e34127d,SILLA ESCOLAR BRAZO IZQUIERDO,NaN,NaN,NaN,All,NaN,None,NaN
24,__export__.product_template_219314_bdfe81d3,SILLA GIRATORIA CON BRAZO,NaN,NaN,NaN,All,NaN,None,NaN


##### Archivo 'ProveedoresParaActualizar.csv' generado

Contiene 5216 productos para actualizar


### Identificación de proveedores no existentes en Odoo

In [6]:
display(Markdown("#### Análisis de nuevos proveedores"))
assigned_sellers = df_odoo_sellers_import['seller_ids'].astype(str).unique()
print(f"Proveedores únicos por asignar: {len(assigned_sellers)}")

df_odoo_sellers = pd.read_csv(raw_path / 'ProveedoresOdoo.csv')
df_odoo_sellers_copy = df_odoo_sellers.copy()
print(f"Proveedores existentes en Odoo: {len(df_odoo_sellers)}")

# Normalizar nombre de proveedores
def normalize_seller_name(name):
    name = str(name).strip()
    name = re.sub(r'^PROV\s*[-]?\s*', '', name, flags=re.IGNORECASE)     # Eliminar prefijo PROV (con o sin espacios/guiones)
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('ASCII')     # Eliminar tildes
    name = re.sub(r'[^\w\s]', '', name)     # Eliminar caracteres no alfanuméricos
    return ' '.join(name.lower().split())

# Normalizar proveedores existentes (crear mapa: normalizado → original)
odoo_sellers_map = {
    normalize_seller_name(name): name
    for name in df_odoo_sellers_copy['name'].astype(str)
}

# Identificar proveedores nuevos
new_odoo_sellers = []
for seller in assigned_sellers:
    seller_str = str(seller).strip()
    # Limpiar caracteres de lista si viene como "['Prov A', 'Prov B']"
    seller_str = seller_str.strip("[]'\"")
    if normalize_seller_name(seller_str) not in odoo_sellers_map:
        new_odoo_sellers.append(seller_str)

new_odoo_sellers = list(set(new_odoo_sellers))

display(Markdown("##### Muestra de proveedores existentes"))
display(df_odoo_sellers.head())

display(Markdown("##### Muestra de proveedores nuevos"))
print(f"Proveedores nuevos a crear: {len(new_odoo_sellers)}")
print(new_odoo_sellers[:5] if new_odoo_sellers else "Ninguno")

if new_odoo_sellers:
    df_new_odoo_sellers = pd.DataFrame({'name': new_odoo_sellers})
    df_new_odoo_sellers.to_csv(processed_path / 'ProveedoresNuevosOdoo.csv', index=False)
    print("\nArchivo 'ProveedoresNuevosOdoo.csv' generado")

    display(Markdown("#### Muestra de normalización para nombres de proveedores"))
    for seller in new_odoo_sellers[:3]:
        print(f"Original: {seller}")
        print(f"Normalizado: {normalize_seller_name(seller)}\n")
else:
    print("\n No hay proveedores nuevos que crear en Odoo")

#### Análisis de nuevos proveedores

Proveedores únicos por asignar: 210
Proveedores existentes en Odoo: 707


##### Muestra de proveedores existentes

,id,name,numeroRUC,is_company
0,__export__.res_partner_3518_510b2f6d,1 (VERIFICAR),NaN,NaN
1,__export__.res_partner_1654_f14b5772,123EDU S A,NaN,True
2,__export__.res_partner_1655_59046e0e,3R GROUP SA,NaN,True
3,__export__.res_partner_1656_460b8507,A M LIBROS,NaN,NaN
4,__export__.res_partner_1657_51f0bc63,A.G. DISPLAY,NaN,True


##### Muestra de proveedores nuevos

Proveedores nuevos a crear: 23
['EBRANARA SA', 'JUAN FRANCISCO GUERRA AGUILAR', 'KEVIN MAN DENG', "LA VICTORIA, S.A. o ALMACEN SUN WAH', 'LA VICTORIA, S.A. o ALMACEN SUN WAH", "REPRESENTACIONES GLOBOLAND SA', 'REPRESENTACIONES GLOBOLAND SA"]

Archivo 'ProveedoresNuevosOdoo.csv' generado


#### Muestra de normalización para nombres de proveedores

Original: EBRANARA SA
Normalizado: ebranara sa

Original: JUAN FRANCISCO GUERRA AGUILAR
Normalizado: juan francisco guerra aguilar

Original: KEVIN MAN DENG
Normalizado: kevin man deng



## ANÁLISIS DE POSIBLES PROVEEDORES A ELIMINAR

In [25]:
FUZZY_THRESHOLD = 85
RUC_COL = 'numeroRUC'
TYPE_COL = 'is_company'  

# NORMALIZACIÓN
def normalize(name):
    name = re.sub(r'^PROV\s*[-]?\s*', '', str(name).strip(), flags=re.IGNORECASE)
    name = unicodedata.normalize('NFKD', name).encode('ASCII', 'ignore').decode('ASCII')
    name = re.sub(r'[^\w\s]', '', name)
    return ' '.join(name.lower().split())

df_odoo_sellers_copy['name_norm'] = df_odoo_sellers_copy['name'].apply(normalize)
df_odoo_sellers_copy['tipo'] = df_odoo_sellers_copy[TYPE_COL].map({True: 'Empresa', False: 'Persona'}).fillna('Persona')  if TYPE_COL in df_odoo_sellers_copy.columns else 'Desconocido'

# DUPLICADOS POR FUZZY
names_unique = df_odoo_sellers_copy[['name', 'name_norm']].drop_duplicates(subset='name_norm')
norm_list = names_unique['name_norm'].tolist()
orig_list = names_unique['name'].tolist()

fuzzy_rows, assigned = [], set()
for i, (ni, oi) in enumerate(zip(norm_list, orig_list)):
    if ni in assigned:
        continue
    group = [(nj, oj) for j, (nj, oj) in enumerate(zip(norm_list, orig_list))
             if i != j and nj not in assigned and fuzz.token_sort_ratio(ni, nj) >= FUZZY_THRESHOLD]
    if group:
        fuzzy_rows.append({'nombre_original': oi, 'duplicados': [o for _, o in group],
                           'cantidad': len(group) + 1})
        assigned.update([ni] + [n for n, _ in group])

df_fuzzy = pd.DataFrame(fuzzy_rows).sort_values('cantidad', ascending=False)
display(Markdown("### Duplicados por similitud de nombre"))
display(df_fuzzy)
df_fuzzy.to_csv(processed_path / 'DuplicadosFuzzy.csv', index=False)

# DUPLICADOS POR RUC
if RUC_COL in df_odoo_sellers_copy.columns:
    df_odoo_sellers_copy['ruc_norm'] = (
        df_odoo_sellers_copy[RUC_COL]
        .astype(str)
        .str.replace(r'[^0-9]', '', regex=True)  # solo dígitos
    )
    invalids = {'', '0'}
    ruc_valid = df_odoo_sellers_copy[~df_odoo_sellers_copy['ruc_norm'].isin(invalids)]

    if not ruc_valid.empty:
        df_ruc_dups = (
            ruc_valid.groupby('ruc_norm')
            .agg(
                nombres=('name', list),
                tipo=('tipo', lambda x: list(x.unique())),
                cantidad=('name', 'count')
            )
            .reset_index()
        )
        df_ruc_dups = df_ruc_dups[df_ruc_dups['cantidad'] > 1].sort_values('cantidad', ascending=False)
        display(Markdown(f"### Duplicados por RUC: {len(df_ruc_dups)}"))
        display(df_ruc_dups)

# PALABRAS CLAVE EN NOMBRE
keywords = ['EVALUAR', 'DUPLICADO', 'ELIMINAR']
for kw in keywords:
    mask = df_odoo_sellers_copy['name'].str.contains(kw, case=False, na=False)
    display(Markdown(f"### Contienen '{kw}': {mask.sum()} proveedores"))
    display(df_odoo_sellers_copy[mask][['name', 'tipo']])

df_odoo_sellers_copy['keyword_detectada'] = df_odoo_sellers_copy['name'].apply(
    lambda x: next((kw for kw in keywords if kw.lower() in x.lower()), None)
)
df_keywords = df_odoo_sellers_copy[df_odoo_sellers_copy['keyword_detectada'].notna()][['name', 'tipo', 'keyword_detectada']]

# PROVEEDORES MEZCLADOS ("/" o "o/ó")
def is_mixed(name):
    if '/' in str(name):
        return True
    parts = re.split(r'\s+[oó]\s+', str(name), flags=re.IGNORECASE)
    return len(parts) > 1 and all(len(p.strip()) >= 3 for p in parts)

def get_separator(name):
    if '/' in str(name):
        return '/'
    return 'o/ó' if is_mixed(name) else None

df_odoo_sellers_copy['mezclado'] = df_odoo_sellers_copy['name'].apply(is_mixed)
df_odoo_sellers_copy['separador'] = df_odoo_sellers_copy['name'].apply(get_separator)
df_odoo_sellers_copy['partes'] = df_odoo_sellers_copy.apply(
    lambda r: [p.strip() for p in re.split(r'/|\s+[oó]\s+', r['name'], flags=re.IGNORECASE)]
    if r['mezclado'] else None, axis=1
)

df_mixed = df_odoo_sellers_copy[df_odoo_sellers_copy['mezclado']][['name', 'tipo', 'separador', 'partes']]
display(Markdown(f"### Proveedores mezclados: {len(df_mixed)}"))
display(df_mixed)
df_mixed.to_csv(processed_path / 'ProveedoresMezclados.csv', index=False)

# PROVEEDORES CLASIFICADOS EN EMPRESA/PERSONA
SA_PATTERN = r'\bsa\b'
PROV_PATTERN = r'^prov[-\s]'

df_odoo_sellers_copy['tiene_sa'] = df_odoo_sellers_copy['name_norm'].str.contains(SA_PATTERN, regex=True)
df_odoo_sellers_copy['tiene_prov'] = df_odoo_sellers_copy['name'].str.contains(PROV_PATTERN, case=False, regex=True, na=False)
df_odoo_sellers_copy['sin_ruc'] = df_odoo_sellers_copy['ruc_norm'].isin(invalidos) if RUC_COL in df.columns else None
df_odoo_sellers_copy['es_fuzzy_dup'] = df_odoo_sellers_copy['name'].isin([n for row in fuzzy_rows for n in [row['nombre_original']] + row['duplicados']])
df_odoo_sellers_copy['es_ruc_dup'] = df_odoo_sellers_copy['name'].isin([n for row in df_ruc.itertuples() for n in row.duplicados]) if RUC_COL in df.columns else False

cols_maestra = ['name', 'tipo', 'tiene_sa', 'tiene_prov', 'mezclado','sin_ruc', 'es_fuzzy_dup', 'es_ruc_dup', 'keyword_detectada']
df_maestra = df_odoo_sellers_copy[cols_maestra].copy()
df_maestra.loc[df_maestra['tipo'] == 'Empresa', 'tiene_prov'] = False

display(Markdown("### Proveedores clasificados (Empresa/Persona)"))
display(df_maestra.head(20))
df_maestra.to_csv(processed_path / 'ProveedoresClasificados.csv', index=False)

# RESUMEN EJECUTIVO
resumen = pd.DataFrame({
    'Categoría': [
        'Total proveedores',
        'Empresas', 'Personas',
        'Duplicados fuzzy (grupos)',
        'Duplicados por RUC (grupos)',
        'Mezclados con / o "o/ó"',
        "Contienen 'EVALUAR'", "Contienen 'DUPLICADO'", "Contienen 'ELIMINAR'",
        'Personas con prefijo PROV-',
        'Personas con SA en nombre',
        'Sin RUC',
    ],
    'Cantidad': [
        len(df_odoo_sellers_copy),
        (df_odoo_sellers_copy['tipo'] == 'Empresa').sum(), (df_odoo_sellers_copy['tipo'] == 'Persona').sum(),
        len(df_fuzzy),
        len(df_ruc) if RUC_COL in df_odoo_sellers_copy.columns else 'N/A',
        len(df_mixed),
        *[(df_odoo_sellers_copy['name'].str.contains(kw, case=False, na=False)).sum() for kw in keywords],
        (df_odoo_sellers_copy[df_odoo_sellers_copy['tipo'] == 'Persona']['tiene_prov']).sum(),
        (df_odoo_sellers_copy[df_odoo_sellers_copy['tipo'] == 'Persona']['tiene_sa']).sum(),
        df_odoo_sellers_copy['sin_ruc'].sum() if RUC_COL in df_odoo_sellers_copy.columns else 'N/A',
    ]
})
display(Markdown("## Resumen del diagnóstico"))
display(resumen)

### Duplicados por similitud de nombre

,nombre_original,duplicados,cantidad
1,DISTRIBUIDORA FT SA,"[DISTRIBUIDORA G&G SA, DISTRIBUIDORA LILY S.A....",4
2,DISTRIBUIDORA MADIX,"[DISTRIBUIDORA MARINER, DISTRIBUIDORA MEDICLA]",3
0,DISCAYCA PANAMA SA,[DISPLAY Panamá SA],2
3,DISTRIBUIDORA ROCKO,[DISTRIBUIDORA ROSARIO],2
4,DO IT CENTER,[PROV-DOIT CENTER],2
5,IMPORTADORA CENTRAL SA,[IMPORTADORA HENRY S.A.],2
6,KADI INTERNATIONAL SA,[POFI INTERNATIONAL SA],2
7,PRODUCTOS LILI SA,[PROV-PRODUCTOS IBIS SA],2
8,PROV-SANTILLANA COLEGIOS FERIAS,[sntillana feria colegio],2
9,PROV-SUSAETA EDICIONES PANAMA SA,"[SUSAETA EDICIONES PANAMA, S.A. (ELIMINAR)]",2


### Duplicados por RUC: 0

,ruc_norm,nombres,tipo,cantidad


### Contienen 'EVALUAR': 5 proveedores

...

,name,tipo,separador,partes
24,"ALHER PAPELES/ALHER INVESTMENT PANAMA,S,A.",Empresa,/,"[ALHER PAPELES, ALHER INVESTMENT PANAMA,S,A.]"
30,ALMACEN HAPPY BIRTHDAY ó Alberto Loo Cheung,Empresa,o/ó,"[ALMACEN HAPPY BIRTHDAY, Alberto Loo Cheung]"
32,ALMACEN ROSITA S.A/ALMACEN WENDY,Empresa,/,"[ALMACEN ROSITA S.A, ALMACEN WENDY]"
33,ALMACEN TIN TON/SHOW MING LAU CHONG,Empresa,/,"[ALMACEN TIN TON, SHOW MING LAU CHONG]"
34,"ALMACEN Y FARMACIA LEE/GRUPO LAU & LIN, S.A.",Empresa,/,"[ALMACEN Y FARMACIA LEE, GRUPO LAU & LIN, S.A.]"
41,ANDREITA Y CHEPITO ó YODALIS CECILIA LEDEZMA ARCE,Persona,o/ó,"[ANDREITA Y CHEPITO, YODALIS CECILIA LEDEZMA A..."
72,BLANCA E. AGUILAR R./MAURO ZUNIGA ARAUZ,Persona,/,"[BLANCA E. AGUILAR R., MAURO ZUNIGA ARAUZ]"
84,CARLOS J DONDERIS o Marketing & Editorial,Persona,o/ó,"[CARLOS J DONDERIS, Marketing & Editorial]"
90,CENTRAL OUTLE/LAYTH NASSIM RAFAEL JABER ABU,Persona,/,"[CENTRAL OUTLE, LAYTH NASSIM RAFAEL JABER ABU]"
108,CONFECCIONES D-BB o Tirza Gonzalez,Persona,o/ó,"[CONFECCIONES D-BB, Tirza Gonzalez]"


### Proveedores clasificados (Empresa/Persona)

,name,tipo,tiene_sa,tiene_prov,mezclado,sin_ruc,es_fuzzy_dup,es_ruc_dup,keyword_detectada
0,1 (VERIFICAR),Persona,False,False,False,True,False,False,None
1,123EDU S A,Empresa,False,False,False,True,False,False,None
2,3R GROUP SA,Empresa,True,False,False,True,False,False,None
3,A M LIBROS,Persona,False,False,False,True,False,False,None
4,A.G. DISPLAY,Empresa,False,False,False,True,False,False,None
5,ABC TRAIDING HOLDING SA,Empresa,True,False,False,True,False,False,None
6,ABDEL ANTONIO MARTINEZ JUSTAVINO,Persona,False,False,False,False,False,False,None
7,ABERNATHY S A,Empresa,False,False,False,True,False,False,None
8,ACADEMIA PROY FOLKLORE,Persona,False,False,False,True,False,False,None
9,ACCESORIOS CENTER IVAN LU,Empresa,False,False,False,True,False,False,None


## Resumen del diagnóstico

,Categoría,Cantidad
0,Total proveedores,707
1,Empresas,538
2,Personas,169
3,Duplicados fuzzy (grupos),11
4,Duplicados por RUC (grupos),0
5,"Mezclados con / o ""o/ó""",38
6,Contienen 'EVALUAR',5
7,Contienen 'DUPLICADO',2
8,Contienen 'ELIMINAR',1
9,Personas con prefijo PROV-,18
